In [0]:
def numeros_generador():
    for i in range(1, 6):
        yield i

g = numeros_generador()
print(g)

In [0]:
for numero in g:
    print(numero)

In [0]:
for numero in g:
    print(numero)

# Decoradores, Context Managers y Typing (para Data Engineering)

Este notebook explica **3 herramientas muy usadas** en proyectos reales:

- **Decoradores**: agregar comportamiento (logs, timing, retry) sin tocar la lógica principal.
- **Context Managers**: asegurar apertura/cierre de recursos con `with` (archivos, conexiones, temporales).
- **Typing**: declarar tipos para que el código sea más claro y mantenible (contratos de datos).

**Objetivo**: entender el concepto + practicar con ejemplos simples que van subiendo el nivel.


## 1) Decoradores

Un **decorador** es una función que **recibe una función** y devuelve otra función que la **envuelve**.

Idea mental:

- La función original es el “core”.
- El decorador agrega cosas **antes** y/o **después** del core.
- Así evitas repetir el mismo código en 10 funciones distintas.

> Nota: muchas veces a la función interna se le llama `wrapper` (envoltura).


### 1.1 Ejemplo mínimo (antes / después)

Lo típico: imprimir algo antes y después de ejecutar la función original.


In [0]:
def decorador(f):
    # f es la función original (la que queremos envolver)
    def wrapper():
        # Esto se ejecuta ANTES del core
        print("Antes")

        # Aquí se ejecuta la función original
        f()

        # Esto se ejecuta DESPUÉS del core
        print("Después")

    # OJO: devolvemos la función wrapper (no la ejecutamos aquí)
    return wrapper


@decorador
def hola():
    # Esta es la función "core"
    print("Hola!")


# Cuando llamamos hola(), en realidad se llama wrapper()
hola()


### 1.2 ¿Qué pasa internamente con `@decorador`?

Cuando escribes:

```python
@decorador
def hola():
    ...
```

Python hace (internamente):

```python
hola = decorador(hola)
```

O sea, `hola` deja de apuntar a la función original y pasa a apuntar a `wrapper`.


### 1.3 Ejemplo con retorno (decorador que modifica el resultado)

Aquí el decorador captura lo que retorna la función y lo modifica.


In [0]:
def duplicar_resultado(f):
    def wrapper():
        # Ejecuta la función original
        r = f()

        # Modifica el resultado
        return r * 2

    return wrapper


@duplicar_resultado
def numero():
    return 10


print(numero())  # 20


### 1.4 `*args` y `**kwargs` (clave para decoradores)

Un decorador "general" debe servir para funciones con distintos parámetros.

- `*args` captura **argumentos posicionales** (van por posición). Se guarda como **tupla**.
- `**kwargs` captura **argumentos con nombre** (clave=valor). Se guarda como **diccionario**.

Ejemplo: si llamas `f(1, 2, x=10)` entonces:
- `args == (1, 2)`
- `kwargs == {'x': 10}`


In [0]:
def mostrar_args_kwargs(*args, **kwargs):
    print("args =", args)      # tupla
    print("kwargs =", kwargs)  # dict


mostrar_args_kwargs(1, 2, x=10, y=20)


Ahora un decorador que funciona con *cualquier* firma de función:

In [0]:
def log_args(f):
    def wrapper(*args, **kwargs):
        # Log simple de parámetros recibidos
        print("Llamando a:", f.__name__)
        print("args:", args)
        print("kwargs:", kwargs)

        # Reenvía EXACTAMENTE los mismos parámetros a la función original
        return f(*args, **kwargs)

    return wrapper


@log_args
def suma(a, b):
    return a + b


@log_args
def saludar(nombre, saludo="Hola"):
    return f"{saludo}, {nombre}!"


print(suma(3, 4))
print(saludar("Luis"))
print(saludar("Ana", saludo="Buenas"))


### 1.5 `range()` dentro de un decorador (reintentos / repetición)

`range(n)` genera `0..n-1`. Es perfecto para intentar n veces.

Ejemplo: repetir una función `n` veces (sin parámetros, para que sea simple).


In [0]:
def repetir(n):
    # Este es un DECORADOR CON PARÁMETRO (n)
    def decorador(f):
        def wrapper():
            # range(n) -> 0..n-1 (n veces)
            for _ in range(n):
                f()
        return wrapper
    return decorador


@repetir(3)
def beep():
    print("beep")


beep()  # imprime "beep" 3 veces


### 1.6 Mini retry (reintentos) para errores temporales

En Data Engineering es común que:
- una API falle 1 vez,
- haya un timeout,
- o una lectura sea intermitente.

Este patrón intenta `n` veces antes de rendirse.


In [0]:
import time

def retry(n, espera_s=1.0):
    # Decorador con parámetros: cantidad de intentos y tiempo de espera
    def decorador(f):
        def wrapper(*args, **kwargs):
            for intento in range(1, n + 1):
                try:
                    print(f"Intento {intento}...")
                    return f(*args, **kwargs)  # si funciona, SALIMOS por return
                except Exception as e:
                    print("Falló:", e)
                    time.sleep(espera_s)

            # Si llegamos aquí, fallaron todos los intentos
            raise Exception("Falló después de varios intentos")
        return wrapper
    return decorador


contador = {"veces": 0}

@retry(n=5, espera_s=0.3)
def llamar_api_fake():
    # Simula que falla 2 veces y luego funciona
    contador["veces"] += 1
    if contador["veces"] < 3:
        raise Exception("API caída (simulada)")
    return "OK"


print("Resultado final:", llamar_api_fake())


### ✅ Ejercicios (Decoradores) — 4 ejercicios progresivos

1) **`@anunciar`**: imprime `"Ejecutando <nombre_funcion>"` antes de correr la función.  
   *Pista:* usa `f.__name__`.

2) **`@doble`**: multiplica por 2 el retorno de una función que devuelve un número.

3) **`@repetir(n)`**: repite la ejecución `n` veces (similar a lo visto).  
   *Pista:* usa `for _ in range(n)`.

4) **`@timer`**: imprime cuánto demoró la función.  
   *Pista:* usa `time.time()` antes y después.

Abajo hay esqueletos con ideas.


In [0]:
# EJERCICIO 1: @anunciar
def anunciar(f):
    # IDEA: define wrapper() que imprima el nombre y luego llame a f()
    def wrapper(*args, **kwargs):
        # TODO: print(f"Ejecutando {f.__name__}")
        # TODO: return f(*args, **kwargs)
        return f(*args, **kwargs)
    return wrapper


@anunciar
def tarea_simple():
    print("Haciendo algo...")

# tarea_simple()


In [0]:
tarea_simple()

In [0]:
# EJERCICIO 2: @doble (modifica retorno numérico)
def doble(f):
    def wrapper(*args, **kwargs):
        # IDEA:
        # r = f(*args, **kwargs)
        # return r * 2
        return f(*args, **kwargs)
    return wrapper


@doble
def leer_registros():
    return 7

# print(leer_registros())


In [0]:
# EJERCICIO 3: @repetir(n) (decorador con parámetro)
def repetir_n(n):
    # IDEA: repetir_n(n) devuelve decorador; decorador recibe f y devuelve wrapper
    def decorador(f):
        def wrapper(*args, **kwargs):
            # TODO: for _ in range(n): f(*args, **kwargs)
            pass
        return wrapper
    return decorador


@repetir_n(2)
def ping():
    print("ping")

# ping()


In [0]:
# EJERCICIO 4: @timer (medir duración)
import time

def timer(f):
    def wrapper(*args, **kwargs):
        # IDEA:
        # inicio = time.time()
        # r = f(*args, **kwargs)
        # fin = time.time()
        # print(f"{f.__name__} tomó {fin - inicio:.2f}s")
        # return r
        return f(*args, **kwargs)
    return wrapper


@timer
def paso_etl_fake():
    time.sleep(0.2)
    return "listo"

# print(paso_etl_fake())


## 2) Context Managers (`with`)

Un **context manager** te asegura:

- Preparación (setup) al entrar al bloque `with`
- Limpieza (cleanup) al salir del bloque (aunque haya errores)

En Data Engineering esto evita:
- archivos abiertos sin cerrar
- conexiones a DB colgadas
- temporales sin borrar

Dos formas comunes:
1) Usar uno ya hecho (`open`, conexiones, etc.)
2) Crear uno custom con `__enter__` y `__exit__`


### 2.1 Uso básico con archivos

`with open(...) as f:` asegura que el archivo se cierre al salir del bloque.


In [0]:
# Escribe un archivo y se cierra automáticamente al salir del bloque
with open("demo_ctx.txt", "w", encoding="utf-8") as f:
    f.write("Hola desde with!\n")

# Lee el archivo y se cierra automáticamente
with open("demo_ctx.txt", "r", encoding="utf-8") as f:
    contenido = f.read()

print(contenido)


### 2.2 Context manager custom (con clase)

- `__enter__`: qué hago al entrar (ej. “abrir conexión”)
- `__exit__`: qué hago al salir (ej. “cerrar conexión”)

`__exit__` recibe info del error si ocurrió algo dentro del `with`.


In [0]:
class ConexionFakeDB:
    def __enter__(self):
        # Setup (ej. abrir conexión)
        print("✅ Conectando a DB (fake)")
        self.conectado = True
        return self  # lo que recibirá el 'as conn'

    def __exit__(self, exc_type, exc_value, traceback):
        # Cleanup (ej. cerrar conexión)
        self.conectado = False
        print("🧹 Cerrando conexión DB (fake)")

        # Si retornas True, "suprime" la excepción.
        # Aquí retornamos False/None para NO ocultar errores.
        return False

    def query(self, sql: str) -> str:
        if not self.conectado:
            raise RuntimeError("No hay conexión")
        return f"Resultado fake para: {sql}"


with ConexionFakeDB() as conn:
    r = conn.query("SELECT 1")
    print(r)


### 2.3 `with` + error: el cleanup siempre pasa

Si hay error dentro del `with`, igual se ejecuta `__exit__`.


In [0]:
class ControlErrores:
    def __enter__(self):
        print("Inicio del bloque")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is not None:
            print("⚠️ Hubo un error:", exc_value)
        print("Fin del bloque (cleanup)")
        return False  # no ocultar el error

try:
    with ControlErrores():
        print("Dentro del with...")
        raise ValueError("Algo falló")
except ValueError:
    print("El error se propagó (normal)")


### 2.4 Context manager para temporales (idea típica en ETL)

Creas un temporal y te aseguras de borrarlo al final.


In [0]:
import os
from pathlib import Path

class TempFile:
    def __init__(self, path: str):
        self.path = Path(path)

    def __enter__(self):
        print("Creando temporal:", self.path)
        self.path.write_text("temporal\n", encoding="utf-8")
        return self.path

    def __exit__(self, exc_type, exc_value, traceback):
        if self.path.exists():
            print("Borrando temporal:", self.path)
            os.remove(self.path)
        return False

with TempFile("tmp_etl.txt") as p:
    print("Usando temporal:", p)
    print(p.read_text(encoding="utf-8"))


### ✅ Ejercicios (Context Managers) — 4 ejercicios progresivos

1) Crear `PrintBlock` que imprima **"Inicio"** al entrar y **"Fin"** al salir.

2) Crear `SafeDivision` que muestre el error si pasa algo dentro del `with` (por ejemplo división entre 0).
   *Pista:* `exc_type` / `exc_value`.

3) Crear `FakeConnection` que imprima "Conectando" / "Cerrando" y tenga método `.run(step)`.

4) Crear `TimerBlock` que mida cuánto duró el bloque `with`.
   *Pista:* guarda `time.time()` en `__enter__` y calcula en `__exit__`.


In [0]:
# EJERCICIO 1: PrintBlock
class PrintBlock:
    def __enter__(self):
        # TODO: print("Inicio")
        pass

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: print("Fin")
        return False


# with PrintBlock():
#     print("Dentro del bloque")


In [0]:
# EJERCICIO 2: SafeDivision (idea simple)
class SafeDivision:
    def __enter__(self):
        # (opcional) 
        # print("Preparando división")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: si exc_type no es None, imprimir exc_value
        return False


# try:
#     with SafeDivision():
#         print(10 / 0)
# except ZeroDivisionError:
#     print("Se propagó el error")


In [0]:
# EJERCICIO 3: FakeConnection
class FakeConnection:
    def __enter__(self):
        # TODO: print("Conectando...")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: print("Cerrando...")
        return False

    def run(self, step: str):
        # TODO: print(f"Ejecutando: {step}")
        pass


# with FakeConnection() as c:
#     c.run("extract")
#     c.run("transform")
#     c.run("load")


In [0]:
# EJERCICIO 4: TimerBlock
import time

class TimerBlock:
    def __enter__(self):
        # TODO: self.inicio = time.time()
        pass

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: fin = time.time(); print duración
        return False


# with TimerBlock():
#     time.sleep(0.2)


## 3) Typing (`typing`)

Typing te permite declarar **qué tipos esperas** en parámetros y retornos.

Beneficios prácticos:
- Código más legible (“contratos”)
- Mejor autocompletado
- Menos bugs por formatos inesperados (especialmente en equipos)

⚠️ Typing no “detiene” errores en runtime por sí solo, pero ayuda a detectarlos antes (linters/type checkers).


### 3.1 Tipos básicos

In [0]:
def suma(a: int, b: int) -> int:
    # a y b deberían ser enteros
    return a + b

print(suma(3, 5))


### 3.2 Listas y diccionarios tipados

In [0]:
def promedio(numeros: list[int]) -> float:
    # numeros: lista de enteros
    return sum(numeros) / len(numeros)

print(promedio([1, 2, 3, 4]))


In [0]:
def total_por_ciudad(ventas: dict[str, int]) -> int:
    # ventas: {"Lima": 10, "Cusco": 5, ...}
    return sum(ventas.values())

print(total_por_ciudad({"Lima": 10, "Cusco": 5}))


### 3.3 `Optional` (cuando puede venir `None`)

In [0]:
from typing import Optional

def saludar(nombre: Optional[str]) -> str:
    # nombre puede ser str o None
    if nombre is None:
        return "Hola, invitado"
    return f"Hola, {nombre}"

print(saludar(None))
print(saludar("Luis"))


### 3.4 `Iterable` / `Iterator` (útil para flujos)

In [0]:
from typing import Iterable, Iterator

def filtrar_pares(valores: Iterable[int]) -> Iterator[int]:
    # Devuelve un generador (iterator) con pares
    for v in valores:
        if v % 2 == 0:
            yield v

stream = filtrar_pares([1, 2, 3, 4, 5, 6])
print(next(stream))   # 2
print(list(stream))   # [4, 6] (ya consumimos el 2)


### 3.5 `TypedDict` (estructura tipo “registro / fila”)

In [0]:
from typing import TypedDict

class RegistroVenta(TypedDict):
    id: int
    producto: str
    monto: float

def procesar_venta(v: RegistroVenta) -> float:
    # Asumimos que v tiene id, producto y monto
    return v["monto"] * 1.18  # ejemplo: impuesto

venta: RegistroVenta = {"id": 1, "producto": "SOAT", "monto": 100.0}
print(procesar_venta(venta))


### ✅ Ejercicios (Typing) — 4 ejercicios progresivos

1) `mayusculas(nombres: list[str]) -> list[str]`  
   *Idea:* `[n.upper() for n in nombres]`

2) `sumar_dict(d: dict[str, int]) -> int`  
   *Idea:* `sum(d.values())`

3) `parse_edad(x: Optional[str]) -> Optional[int]`  
   *Idea:* si x es None retorna None; si no, `int(x)`

4) `TypedDict` `RegistroUser` con `id:int`, `email:str`, `activo:bool` y función `is_activo(u: RegistroUser) -> bool`.


In [0]:
# EJERCICIO 1
def mayusculas(nombres: list[str]) -> list[str]:
    # TODO: retornar lista con todos en mayúscula
    # IDEA: return [n.upper() for n in nombres]
    pass

# print(mayusculas(["luis", "ana"]))


In [0]:
# EJERCICIO 2
def sumar_dict(d: dict[str, int]) -> int:
    # TODO
    # IDEA: return sum(d.values())
    pass

# print(sumar_dict({"a": 1, "b": 2}))


In [0]:
# EJERCICIO 3
from typing import Optional

def parse_edad(x: Optional[str]) -> Optional[int]:
    # TODO
    # IDEA:
    # if x is None: return None
    # return int(x)
    pass

# print(parse_edad(None))
# print(parse_edad("30"))


In [0]:
# EJERCICIO 4
from typing import TypedDict

class RegistroUser(TypedDict):
    # TODO: id, email, activo
    pass

def is_activo(u: RegistroUser) -> bool:
    # TODO: return u["activo"]
    pass

# user: RegistroUser = {"id": 1, "email": "a@b.com", "activo": True}
# print(is_activo(user))


---
## Cierre

- **Decoradores**: comportamiento reusable (logs, timer, retry) sin duplicar código.
- **Context Managers**: cleanup garantizado (archivos, conexiones, temporales).
- **Typing**: contratos claros para trabajar con datos cambiantes.
